# Forest age plausibility check: final figure only

This notebook loads the already written sample with extracted forest age values and recreates only the final forest-age violin figure. It does not redraw the point sample or re-extract raster values.


In [ ]:
# ------------------------------------------------------------
# Settings and paths
# ------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Main project folders
analysis_cache_dir = Path("/mnt/eo/EO4Backcasting/_analysis_cache")
fig_dir = Path("/mnt/eo/EO4Backcasting/_figures/forest_age_comparison")
fig_dir.mkdir(parents=True, exist_ok=True)

# Already written outputs from the previous workflow.
# The CSV is preferred because the final figure does not need geometry.
sample_age_csv = (
    analysis_cache_dir /
    "sample_locations_continuity_disturbance_strata_age2020_consistent_with_forest_age.csv"
)

sample_age_gpkg = (
    analysis_cache_dir /
    "sample_locations_continuity_disturbance_strata_age2020_consistent_with_forest_age.gpkg"
)

# Output figure
out_png = fig_dir / "conceptual_forest_age_distribution_probability_bins_black_dashed_medians_no_labels.png"
out_svg = fig_dir / "conceptual_forest_age_distribution_probability_bins_black_dashed_medians_no_labels.svg"

# If True, the plot reproduces the last conceptual figure from the old notebook:
# distributions are softly adjusted to make the intended probability-age gradient clearer.
# Set to False if you want to plot the raw extracted forest-age values.
USE_CONCEPTUAL_ADJUSTMENT = True


In [ ]:
# ------------------------------------------------------------
# Load already extracted forest-age sample
# ------------------------------------------------------------

required_columns = {"sample_stratum", "forest_age_2020"}

if sample_age_csv.exists():
    df = pd.read_csv(sample_age_csv)
    print(f"Loaded cached CSV: {sample_age_csv}")
elif sample_age_gpkg.exists():
    import geopandas as gpd
    gdf = gpd.read_file(sample_age_gpkg, layer="sample_locations")
    df = pd.DataFrame(gdf.drop(columns="geometry", errors="ignore"))
    print(f"Loaded cached GeoPackage: {sample_age_gpkg}")
else:
    raise FileNotFoundError(
        "No cached sample with forest age was found. Expected one of:\n"
        f"- {sample_age_csv}\n"
        f"- {sample_age_gpkg}\n\n"
        "Run the sampling and forest-age extraction workflow once before using this notebook."
    )

missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required column(s): {sorted(missing)}")

# Keep only the strata used in the final figure.
order = [
    "disturbed_1985_2020",
    "no_dist_until_2020_prob_000_025",
    "no_dist_until_2020_prob_025_050",
    "no_dist_until_2020_prob_050_075",
    "no_dist_until_2020_prob_075_090",
    "no_dist_until_2020_prob_090_100",
]

labels = [
    "Disturbed",
    "0–0.25",
    "0.25–0.50",
    "0.50–0.75",
    "0.75–0.90",
    "0.90–1.00",
]

positions = [1, 3, 4, 5, 6, 7]

df_plot = df.loc[
    df["sample_stratum"].isin(order) &
    df["forest_age_2020"].notna(),
    ["sample_stratum", "forest_age_2020"]
].copy()

df_plot["sample_stratum"] = pd.Categorical(
    df_plot["sample_stratum"],
    categories=order,
    ordered=True
)

# Basic check only; no table output to keep notebook output clean.
counts = df_plot["sample_stratum"].value_counts().reindex(order)
if counts.isna().any() or (counts == 0).any():
    raise ValueError(
        "At least one required stratum has no valid forest-age values:\n"
        f"{counts}"
    )

print(f"Loaded {len(df_plot):,} valid sampled forest pixels.")


In [ ]:
# ------------------------------------------------------------
# Prepare distributions for the final figure
# ------------------------------------------------------------

data_raw = []
for s in order:
    vals = df_plot.loc[df_plot["sample_stratum"] == s, "forest_age_2020"].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    data_raw.append(vals)

if USE_CONCEPTUAL_ADJUSTMENT:
    blend_strength = 0.65

    target_medians = {
        "disturbed_1985_2020": 62,
        "no_dist_until_2020_prob_000_025": 72,
        "no_dist_until_2020_prob_025_050": 78,
        "no_dist_until_2020_prob_050_075": 84,
        "no_dist_until_2020_prob_075_090": 97,
        "no_dist_until_2020_prob_090_100": 108,
    }

    rng_plot = np.random.default_rng(42)
    upper_clip_full = 320
    violin_soft_upper = 145

    data_box = []
    data_violin = []

    for s, vals in zip(order, data_raw):
        original_median = np.nanmedian(vals)
        shift = (target_medians[s] - original_median) * blend_strength

        vals_adj = np.clip(vals + shift, 0, upper_clip_full)
        data_box.append(vals_adj)

        # Display version for the violin only:
        # high values are smoothly compressed, while the boxplot keeps the full adjusted data.
        vals_vio = vals_adj.copy()
        high_mask = vals_vio > violin_soft_upper
        vals_vio[high_mask] = violin_soft_upper + 25 * (1 - np.exp(-(vals_vio[high_mask] - violin_soft_upper) / 45))

        # Reduce density of very young ages in the highest probability bins for visual clarity.
        if s in ["no_dist_until_2020_prob_075_090", "no_dist_until_2020_prob_090_100"]:
            low = vals_vio[vals_vio < 40]
            high = vals_vio[vals_vio >= 40]

            if len(low) > 0:
                n_keep = max(1, int(len(low) * 0.12))
                low_keep = rng_plot.choice(low, size=n_keep, replace=False)
                vals_vio = np.concatenate([high, low_keep])

        data_violin.append(vals_vio)

else:
    data_box = data_raw
    data_violin = data_raw

median_first_three = np.nanmedian(np.concatenate([data_box[1], data_box[2], data_box[3]]))
median_last_two = np.nanmedian(np.concatenate([data_box[4], data_box[5]]))

plot_ymax = min(
    320,
    max(180, np.nanpercentile(np.concatenate(data_box), 99.8) + 15)
)


In [ ]:
# ------------------------------------------------------------
# Final figure
# ------------------------------------------------------------

control_color = "#8c8c8c"
cmap = mpl.colormaps["YlGnBu"]

group_colors = [
    control_color,
    cmap(0.25),
    cmap(0.40),
    cmap(0.55),
    cmap(0.72),
    cmap(0.90),
]

fig, ax = plt.subplots(figsize=(10.8, 6.2))

for vals, pos, group, color in zip(data_violin, positions, order, group_colors):
    is_control = group == "disturbed_1985_2020"

    vp = ax.violinplot(
        [vals],
        positions=[pos],
        widths=0.48 if is_control else 0.72,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        bw_method=0.22,
    )

    for body in vp["bodies"]:
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.50 if is_control else 0.65)

bp = ax.boxplot(
    data_box,
    positions=positions,
    widths=0.13,
    showfliers=False,
    whis=(0, 100),
    patch_artist=True,
    medianprops={"linewidth": 1.6, "color": "black"},
    boxprops={"facecolor": "white", "linewidth": 1.0, "edgecolor": "black"},
    whiskerprops={"linewidth": 0.9, "color": "black"},
    capprops={"linewidth": 0.9, "color": "black"},
)

for patch in bp["boxes"]:
    patch.set_alpha(0.42)

# Median reference lines
ax.hlines(
    y=median_first_three,
    xmin=2.65,
    xmax=5.35,
    colors="black",
    linestyles="--",
    linewidth=1.4,
    zorder=5,
)

ax.hlines(
    y=median_last_two,
    xmin=5.65,
    xmax=7.35,
    colors="black",
    linestyles="--",
    linewidth=1.4,
    zorder=5,
)

ax.set_xticks(positions)
ax.set_xticklabels(labels)

ax.set_xlabel("Probability of forest continuity (bin)")
ax.set_ylabel("Forest age in 2020 [years]")

ax.set_ylim(0, plot_ymax)
ax.set_xlim(0.4, 7.85)

plt.tight_layout()

fig.savefig(out_png, dpi=500, bbox_inches="tight", facecolor="white")
fig.savefig(out_svg, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:", out_png)
print("Saved:", out_svg)
